# Extracting KB

In [ ]:
%pip install langchain langgraph langchain-community ollama pydantic scikit-learn pandas matplotlib seaborn numpy langchain-ollama

import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
import json
from typing import TypedDict, Dict, Any
from langgraph.graph import StateGraph, END
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import HumanMessage, SystemMessage
from typing import TypedDict, Dict, Any, List
from pydantic import BaseModel, Field
from langchain_community.chat_models import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END
from langchain_core.output_parsers import JsonOutputParser



KB_PATH = "./input/KB.md"
PROMPTS_PATH = "./prompts/prompts.json"
ollama_model = "llama3.2"
ollama_model_temperature = 0

# Helper function
def load_json(path: str) -> Dict:
    """Load JSON from file."""
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

## Langgraph setup

In [ ]:
from pydantic import BaseModel, Field
from typing import List

# --- 1. DEFINE STRICT PYDANTIC SCHEMAS (UPDATED TO MATCH IMAGE) ---

class CompanyNorthStar(BaseModel):
    inferred_north_star: str = Field(description="The primary inferred North Star metric")
    justification: str = Field(description="The justification for choosing this metric")
    measurable_proxy: str = Field(description="The measurable proxy for the North Star")

class HookTaxonomy(BaseModel):
    hook_name: str
    description: str

class AllowedToneHookMatrix(BaseModel):
    allowed_tones: List[str]
    disallowed_tones: List[str]
    hook_taxonomy: List[HookTaxonomy] = Field(description="Taxonomy of behavioral hooks")

class FeatureMapping(BaseModel):
    feature_name: str
    lifecycle_stage: str = Field(description="The specific lifecycle stage this feature targets")
    outcome_mapping: str = Field(description="The desired outcome mapping for this feature")

class FeatureGoalMap(BaseModel):
    features: List[FeatureMapping]

# --- 2. HARDCODED PROMPTS & SCHEMA MAPPING ---

PROMPTS = {
    "company_north_star": {
        "system": "You are a product analytics expert. Extract the North Star metric details based on the provided Knowledge Base.",
        "user": "Extract the inferred north star metric, its justification, and a measurable proxy from the following KB:\n\nKB:\n{kb}",
        "schema": CompanyNorthStar
    },
    "allowed_tone_hook_matrix": {
        "system": "You are a behavioral design analyst. Extract tone rules and hook taxonomy based on the provided Knowledge Base.",
        "user": "Extract allowed tones, disallowed tones, and a complete hook taxonomy from the following KB:\n\nKB:\n{kb}",
        "schema": AllowedToneHookMatrix
    },
    "feature_goal_map": {
        "system": "You are a product goal alignment specialist. Extract goal mappings for the provided features based on the Knowledge Base.",
        "user": "For EACH of the following features, map it to a specific user lifecycle stage and its intended outcome mapping.\n\nFeatures to map:\n{features}\n\nUse this KB context to inform mappings:\n{kb}",
        "schema": FeatureGoalMap
    }
}

# --- 3. HARDCODED FEATURES (Extracted from CSV) ---
all_columns = [
    "user_id", "lifecycle_stage", "days_since_signup", "age_band", "region", 
    "sessions_last_7d", "exercises_completed_7d", "streak_current", "coins_balance", 
    "feature_ai_tutor_used", "feature_leaderboard_viewed", "feature_progress_checked", 
    "preferred_hour", "notif_open_rate_30d", "motivation_score", "gamification_propensity", 
    "ai_tutor_propensity", "leaderboard_propensity", "activation_score", "dominant_propensity", 
    "activation_level", "segment_id", "segment_name"
]
exclude = {'user_id', 'segment_id', 'segment_name', 'lifecycle_stage', 'activation_level', 'dominant_propensity'}
filtered_columns = [col for col in all_columns if col not in exclude]

FEATURES_STRING = "\n".join([f"{i+1}. {col}" for i, col in enumerate(filtered_columns)])


# --- 4. LANGGRAPH SETUP ---
class KBState(TypedDict):
    kb_text: str
    prompts: Dict[str, Any]
    outputs: Dict[str, Any]
    prompts_used: Dict[str, Any]

llm = ChatOllama(model=ollama_model, temperature=ollama_model_temperature, format="json")
def save_json(path: str, data: Dict):
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

def run_llm_with_schema(system_prompt: str, user_prompt: str, schema: BaseModel):
    # Initialize the parser with your specific Pydantic model
    parser = JsonOutputParser(pydantic_object=schema)
    
    # Get the strict formatting instructions generated by Pydantic
    format_instructions = parser.get_format_instructions()
    
    # Inject those instructions into the system prompt
    full_system_prompt = f"{system_prompt}\n\n{format_instructions}"
    
    # Invoke the LLM
    res = llm.invoke([
        SystemMessage(content=full_system_prompt), 
        HumanMessage(content=user_prompt)
    ])
    
    try:
        # The parser automatically validates the JSON string against your Pydantic class
        parsed_data = parser.parse(res.content)
        return parsed_data
    except Exception as e:
        print(f"Validation Error: {e}\nRaw Output: {res.content}")
        return {"error": "LLM failed Pydantic validation.", "raw": res.content}
    
def load_inputs(state: KBState):
    if os.path.exists(KB_PATH):
        with open(KB_PATH, "r", encoding="utf-8") as f:
            state["kb_text"] = f.read()
    else:
        state["kb_text"] = "Dummy knowledge base text."
        
    state["prompts"] = PROMPTS
    state["outputs"] = {}
    state["prompts_used"] = {}
    return state

def run_extraction(key: str):
    def node(state: KBState):
        prompt_block = state["prompts"][key]
        user_prompt = prompt_block["user"].replace("{kb}", state["kb_text"])
        
        if key == "feature_goal_map":
            user_prompt = user_prompt.replace("{features}", FEATURES_STRING)
        
        result = run_llm_with_schema(prompt_block["system"], user_prompt, prompt_block["schema"])
        
        state["outputs"][key] = result
        state["prompts_used"][key] = {"system": prompt_block["system"], "user": user_prompt}
        return state
    return node

def save_outputs(state: KBState):
    for key, value in state["outputs"].items():
        save_json(f"./output/{key}.json", value)
        save_json(f"../iteration_0_before_learning/{key}.json", value)
    return state


# --- 5. BUILD AND EXECUTE GRAPH ---
builder = StateGraph(KBState)
builder.add_node("load", load_inputs)

prompt_keys = list(PROMPTS.keys())
prev_node = "load"

for key in prompt_keys:
    current_node = f"extract_{key}"
    builder.add_node(current_node, run_extraction(key))
    builder.add_edge(prev_node, current_node)
    prev_node = current_node

builder.add_node("save", save_outputs)
builder.add_edge(prev_node, "save")
builder.add_edge("save", END)
builder.set_entry_point("load")

graph = builder.compile()

if __name__ == "__main__":
    print("Starting extraction pipeline with updated Pydantic schemas...")
    final_state = graph.invoke({"kb_text": "", "prompts": {}, "outputs": {}, "prompts_used": {}})
    print("Pipeline complete. Check the output folders.")

# Missing data handeling 

In [ ]:
encoded_user_csv_path = "./input/users.csv"
df = pd.read_csv(encoded_user_csv_path)

numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].mean())

df.to_csv("./output/users_imputed.csv", index=False)

# Generating Propensity and Activeness scores

In [ ]:
df = pd.read_csv("./output/users_imputed.csv")

In [ ]:
gamification_score = 0.7 * df['streak_current'] + 0.3 * df['coins_balance']
mu = gamification_score.mean()
sigma = gamification_score.std()
df['gamification_propensity'] = 1 / (1 + np.exp(-(gamification_score - mu) / sigma))
df.head(1)

In [ ]:
ai_tutor_score = 10 * df['feature_ai_tutor_used'] + 1.5 * df['exercises_completed_7d']
mu = ai_tutor_score.mean()
sigma = ai_tutor_score.std()
df['ai_tutor_propensity'] = 1 / (1 + np.exp(-(ai_tutor_score - mu) / sigma))
df.head(1)

In [ ]:
leaderboard_score = 10 * df['feature_leaderboard_viewed'] + 5 * df['motivation_score']
mu = leaderboard_score.mean()
sigma = leaderboard_score.std()
df['leaderboard_propensity'] = 1 / (1 + np.exp(-(leaderboard_score - mu) / sigma))

df.head(1)

In [ ]:
from sklearn.decomposition import PCA

cols = ['sessions_last_7d', 'exercises_completed_7d', 'streak_current', 'notif_open_rate_30d', 'motivation_score']
standardized = pd.DataFrame()
for col in cols:
    standardized[col] = (df[col] - df[col].mean()) / df[col].std()

pca = PCA(n_components=1)
pca.fit(standardized)
weights = {col: pca.components_[0][i] for i, col in enumerate(cols)}

pc1_raw = sum(weights[col] * standardized[col] for col in cols)
pc1_min = pc1_raw.min()
pc1_max = pc1_raw.max()
df['activation_score'] = (pc1_raw - pc1_min) / (pc1_max - pc1_min)

df.head(1)

In [ ]:
df.to_csv("./output/users_propensity_generated.csv", index=False)

# Segmenting users 

In [ ]:
df = pd.read_csv("./output/users_propensity_generated.csv")

df['dominant_propensity'] = df[['gamification_propensity', 'ai_tutor_propensity', 'leaderboard_propensity']].idxmax(axis=1)
df['dominant_propensity'] = df['dominant_propensity'].str.replace('_propensity', '')

def get_activation_level(score):
    if score > 0.7:
        return 'High'
    elif score >= 0.4:
        return 'Medium'
    else:
        return 'Low'

df['activation_level'] = df['activation_score'].apply(get_activation_level)

segment_map = {
    ('gamification', 'High'): ('S1', 'Streak Champions'),
    ('ai_tutor', 'High'): ('S2', 'Power Learners'),
    ('leaderboard', 'High'): ('S3', 'Top Competitors'),
    ('gamification', 'Medium'): ('S4', 'Coin Collectors'),
    ('ai_tutor', 'Medium'): ('S5', 'Steady Students'),
    ('leaderboard', 'Medium'): ('S6', 'Social Observers'),
    ('gamification', 'Low'): ('S7', 'Fading Players'),
    ('ai_tutor', 'Low'): ('S8', 'Stuck Users'),
    ('leaderboard', 'Low'): ('S9', 'Disconnected'),
}

df[['segment_id', 'segment_name']] = df.apply(
    lambda row: pd.Series(segment_map[(row['dominant_propensity'], row['activation_level'])]),
    axis=1
)

df.to_csv("./output/users_segmented.csv", index=False)
df.to_csv("../iteration_0_before_learning/user_segments.csv", index=False)

# Assigning Goals , Sub-Goals for ech segment * lifecycle stage

In [ ]:
import json, os
import pandas as pd
from typing import TypedDict, Dict, Any
from langchain_core.messages import SystemMessage, HumanMessage
from langgraph.graph import StateGraph, END

def run_llm(system_prompt: str, user_prompt: str):
    res = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=user_prompt)])
    try: return json.loads(res.content)
    except json.JSONDecodeError: return {"raw_response": res.content}

# --- 1. Generate Segment × Lifecycle Stage Goals ---
df_segments = pd.read_csv("./output/users_segmented.csv")
feature_goals = load_json("./output/feature_goal_map.json")

lifecycle_context = {
    "trial": {"description": "Early Stage - Building Habits", "goal_modifier": "rapid adoption & habit formation", "focus": "Quick wins and onboarding"},
    "paid": {"description": "Engaged - Maximizing Value", "goal_modifier": "deepening engagement & retention", "focus": "Sustained usage and progression"},
    "churned": {"description": "At-Risk - Re-activation Needed", "goal_modifier": "re-engagement & recovering lost habits", "focus": "Breaking inactivity cycle"},
    "inactive": {"description": "Dormant - Win-Back Campaign", "goal_modifier": "compelling return value proposition", "focus": "Removing friction to return"},
}

feature_goal_map = {
    ptype: {"primary_goal": f"Increase {ptype} engagement", "sub_goals": [f"Drive {ptype} feature usage", f"Improve {ptype} adoption metrics"], "progression_goals": [f"Track daily {ptype} metrics"]}
    for ptype in ["gamification", "ai_tutor", "leaderboard"]
}

segments = sorted(df_segments['segment_id'].unique().tolist())
stages = df_segments['lifecycle_stage'].unique().tolist()
available_stages = (stages + [s for s in ["trial", "paid", "churned", "inactive"] if s not in stages])[:4]

segment_lifecycle_goals = {"combinations": []}
for _, row in df_segments.drop_duplicates(subset=['segment_id']).iterrows():
    base_goals = feature_goal_map.get(row['dominant_propensity'], {})
    
    for stage in available_stages:
        lc_info = lifecycle_context.get(stage, {})
        segment_lifecycle_goals["combinations"].append({
            "combination_id": f"{row['segment_id']}_{stage}",
            "segment_id": row['segment_id'], "segment_name": row['segment_name'],
            "lifecycle_stage": stage, "lifecycle_description": lc_info.get("description", ""),
            "dominant_propensity": row['dominant_propensity'], "activation_level": row['activation_level'],
            "primary_goal": f"{base_goals.get('primary_goal', 'Engagement')} • {lc_info.get('goal_modifier', 'optimize engagement')}",
            "sub_goals": [f"{sg} ({stage.title()} context: {lc_info.get('focus', 'engagement')})" for sg in base_goals.get('sub_goals', [])],
            "day_on_day_progression_goals": [f"{pg} • Track in {stage} phase" for pg in base_goals.get('progression_goals', [])],
            "lifecycle_context": {"stage": stage, **lc_info}
        })

df_goals = pd.DataFrame(segment_lifecycle_goals["combinations"])
df_goals.to_csv("./output/segment_lifecycle_goals.csv", index=False)
df_goals.to_csv("../iteration_0_before_learning/segment_goals.csv", index=False)

# --- 2. Build Segment Profiles (Optimized with Pandas GroupBy) ---
aggs = {
    'segment_name': 'first', 'dominant_propensity': lambda x: x.mode()[0], 'activation_level': lambda x: x.mode()[0],
    'gamification_propensity': 'mean', 'ai_tutor_propensity': 'mean', 'leaderboard_propensity': 'mean', 
    'activation_score': 'mean', 'user_id': 'count'
}
segment_profiles = df_segments.groupby('segment_id').agg(aggs).rename(columns={
    'gamification_propensity': 'avg_gamification', 'ai_tutor_propensity': 'avg_ai_tutor',
    'leaderboard_propensity': 'avg_leaderboard', 'activation_score': 'avg_activation', 'user_id': 'user_count'
}).reset_index().set_index('segment_id', drop=False).to_dict('index')

# --- 3. Setup & Execute Theme Engine ---
class ThemeEngineState(TypedDict):
    segment_profiles: Dict[str, Any]
    tone_hooks: Dict[str, Any]
    outputs: Dict[str, Any]

def load_theme_inputs(state: ThemeEngineState):
    state["segment_profiles"] = segment_profiles
    try:
        hooks = load_json("./output/allowed_tone_hook_matrix.json").get("hook_taxonomy", [])
        state["tone_hooks"] = {"octolysis_hook_mapping": {h.get("hook_name", h.get("name", str(h))): h.get("description", "") for h in hooks if isinstance(h, dict)}}
        if not state["tone_hooks"]["octolysis_hook_mapping"]: raise ValueError
    except Exception:
        state["tone_hooks"] = {"octolysis_hook_mapping": {"Epic Meaning": "Larger purpose", "Accomplishment": "Progress", "Empowerment": "Creativity", "Social Influence": "Relatedness", "Scarcity": "Exclusivity", "Unpredictability": "Surprises", "Loss Avoidance": "FOMO"}}
    return state

def extract_segment_themes(state: ThemeEngineState):
    prompts = load_json(PROMPTS_PATH)["segment_themes"]
    hooks_str = json.dumps(state["tone_hooks"]["octolysis_hook_mapping"], indent=2)
    theme_results = []

    for sid, prof in state["segment_profiles"].items():
        user_prompt = prompts["user"]
        # Safe string replacements
        replacements = {
            "{segment_id}": sid, "{segment_name}": prof['segment_name'], "{dominant_propensity}": prof['dominant_propensity'],
            "{activation_level}": prof['activation_level'], "{avg_gamification}": f"{prof['avg_gamification']:.3f}",
            "{avg_ai_tutor}": f"{prof['avg_ai_tutor']:.3f}", "{avg_leaderboard}": f"{prof['avg_leaderboard']:.3f}",
            "{avg_activation}": f"{prof['avg_activation']:.3f}", "{user_count}": str(prof['user_count']), "{tone_hooks_json}": hooks_str
        }
        for k, v in replacements.items(): user_prompt = user_prompt.replace(k, v)
        
        try:
            theme_results.append(run_llm(prompts["system"], user_prompt))
            print(f"✓ Processed {sid}")
        except Exception as e:
            print(f"✗ Error {sid}: {e}")

    state["outputs"] = {"segment_themes": {"segments": theme_results, "total_segments": len(theme_results)}}
    return state

def save_theme_outputs(state: ThemeEngineState):
    save_json("./output/segment_themes.json", state["outputs"]["segment_themes"])
    return state

theme_builder = StateGraph(ThemeEngineState)
for node in [load_theme_inputs, extract_segment_themes, save_theme_outputs]: theme_builder.add_node(node.__name__, node)
theme_builder.add_edge("load_theme_inputs", "extract_segment_themes")
theme_builder.add_edge("extract_segment_themes", "save_theme_outputs")
theme_builder.add_edge("save_theme_outputs", END)
theme_builder.set_entry_point("load_theme_inputs")

theme_builder.compile().invoke({"segment_profiles": {}, "tone_hooks": {}, "outputs": {}})